In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

In [52]:
torch.manual_seed(42)

d_model = 8
seq_len = 4

# Fake token embeddings (no positional info)
X = torch.randn(1, seq_len, d_model)

In [53]:
X

tensor([[[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784, -1.2345, -0.0431,
          -1.6047],
         [-0.7521,  1.6487, -0.3925, -1.4036, -0.7279, -0.5594, -0.7688,
           0.7624],
         [ 1.6423, -0.1596, -0.4974,  0.4396, -0.7581,  1.0783,  0.8008,
           1.6806],
         [ 1.2791,  1.2964,  0.6105,  1.3347, -0.2316,  0.0418, -0.2516,
           0.8599]]])

In [54]:

W_q = torch.randn(d_model, d_model)
W_k = torch.randn(d_model, d_model)
W_v = torch.randn(d_model, d_model)


In [55]:

def attention(X, W_q, W_k, W_v):
    Q = X @ W_q
    K = X @ W_k
    V = X @ W_v
    scores = Q @ K.transpose(-2, -1) / (d_model ** 0.5)
    weights = torch.softmax(scores, dim=-1)
    return weights @ V

In [56]:
out_original = attention(X, W_q, W_k, W_v)
out_original

tensor([[[-3.6925,  0.7997,  9.4704, -2.5174, -6.2702, -0.8382, -3.9583,
          -3.3199],
         [-1.7831,  5.1673,  3.8020,  2.5630, -3.0025,  1.5952,  0.3802,
           5.1056],
         [-5.2188,  3.3771, -5.2385,  0.9001,  3.2845, -0.4186,  3.6742,
          -0.9937],
         [-5.2142,  3.4009, -5.2754,  0.8955,  3.3392, -0.3942,  3.6949,
          -1.0621]]])

In [57]:
# Shuffle: swap positions 1 and 2
perm = [0, 2, 1, 3]
X_shuffled = X[:, perm, :]

In [58]:
X, X_shuffled

(tensor([[[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784, -1.2345, -0.0431,
           -1.6047],
          [-0.7521,  1.6487, -0.3925, -1.4036, -0.7279, -0.5594, -0.7688,
            0.7624],
          [ 1.6423, -0.1596, -0.4974,  0.4396, -0.7581,  1.0783,  0.8008,
            1.6806],
          [ 1.2791,  1.2964,  0.6105,  1.3347, -0.2316,  0.0418, -0.2516,
            0.8599]]]),
 tensor([[[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784, -1.2345, -0.0431,
           -1.6047],
          [ 1.6423, -0.1596, -0.4974,  0.4396, -0.7581,  1.0783,  0.8008,
            1.6806],
          [-0.7521,  1.6487, -0.3925, -1.4036, -0.7279, -0.5594, -0.7688,
            0.7624],
          [ 1.2791,  1.2964,  0.6105,  1.3347, -0.2316,  0.0418, -0.2516,
            0.8599]]]))

In [59]:
out_shuffled = attention(X_shuffled, W_q, W_k, W_v)
out_original, out_shuffled

(tensor([[[-3.6925,  0.7997,  9.4704, -2.5174, -6.2702, -0.8382, -3.9583,
           -3.3199],
          [-1.7831,  5.1673,  3.8020,  2.5630, -3.0025,  1.5952,  0.3802,
            5.1056],
          [-5.2188,  3.3771, -5.2385,  0.9001,  3.2845, -0.4186,  3.6742,
           -0.9937],
          [-5.2142,  3.4009, -5.2754,  0.8955,  3.3392, -0.3942,  3.6949,
           -1.0621]]]),
 tensor([[[-3.6925,  0.7997,  9.4704, -2.5174, -6.2702, -0.8382, -3.9583,
           -3.3199],
          [-5.2188,  3.3771, -5.2385,  0.9001,  3.2845, -0.4186,  3.6742,
           -0.9937],
          [-1.7831,  5.1673,  3.8020,  2.5630, -3.0025,  1.5952,  0.3802,
            5.1056],
          [-5.2142,  3.4009, -5.2754,  0.8955,  3.3392, -0.3942,  3.6949,
           -1.0621]]]))

In [60]:


# Un-shuffle the output
inv_perm = [0, 2, 1, 3]
out_unshuffled = out_shuffled[:, inv_perm, :]

print('Max difference after un-shuffling:', (out_original - out_unshuffled).abs().max().item())
print('\n→ Outputs are identical (up to float precision).')
print('  Attention cannot tell "dog bites man" from "man bites dog".')

Max difference after un-shuffling: 9.5367431640625e-07

→ Outputs are identical (up to float precision).
  Attention cannot tell "dog bites man" from "man bites dog".


In [ ]:
torch.manual_seed(42)

d_model = 8
seq_len = 4

# Fake token embeddings (no positional info)
X = torch.randn(1, seq_len, d_model)

In [ ]:
perm = [0, 2, 1, 3]
X_shuffled = X[:, perm, :]

In [48]:
a = X @ W_q
b = X @ W_k
R1 = a @ b.transpose(-2, -1)

In [49]:
c = X_shuffled @ W_q
d = X_shuffled @ W_k
R2 = c @ d.transpose(-2, -1)

In [50]:
R1

tensor([[[ 48.3592,  -1.4314,   7.0602,  16.1742],
         [  1.8840,  14.5877, -10.8521, -11.8844],
         [-20.8970,  -3.9824,  16.8504,   5.9619],
         [  7.2162,   3.6702,  49.6052,  35.6275]]])

In [51]:
R2

tensor([[[ 48.3592,   7.0602,  -1.4314,  16.1742],
         [-20.8970,  16.8504,  -3.9824,   5.9619],
         [  1.8840, -10.8521,  14.5877, -11.8844],
         [  7.2162,  49.6052,   3.6702,  35.6275]]])

In [39]:
x = torch.Tensor([[[1,2], [3,4]]])

In [40]:
x

tensor([[[1., 2.],
         [3., 4.]]])

In [41]:
uq = torch.Tensor([[4,5], [6,7]])
uk = torch.Tensor([[8,9], [10,11]])

In [42]:
perm = [1, 0]
xs = x[:, perm, :]

In [43]:
xs

tensor([[[3., 4.],
         [1., 2.]]])

In [44]:
a = x @ uq
b = x @ uk
R1 = a @ b.transpose(-2, -1)

In [45]:
c = xs @ uq
d = xs @ uk
R2 = c @ d.transpose(-2, -1)

In [46]:
R1

tensor([[[1037., 2373.],
         [2341., 5357.]]])

In [47]:
R2

tensor([[[5357., 2341.],
         [2373., 1037.]]])